In [ ]:
column = "llm_specific_processes"
file = "skills.csv"

import pandas as pd

df = pd.read_csv(file)
res = df[column].dropna().unique()  # list of strings, each string is a list
import ast

res = [ast.literal_eval(x) for x in res]  # list of lists
res = [", ".join(x) for x in res]  # list of strings
# drop empty
res = [x for x in res if x.strip()]
res = list(set(res))  # unique values
# res = [item for sublist in res for item in sublist]  # flatten the list
# res = list(set(res))  # unique values
print(len(res))
print(res[:10])

In [2]:
from openai import OpenAI

client = OpenAI()
embeddings = client.embeddings.create(input=res, model="text-embedding-3-small")
embeddings = [e.embedding for e in embeddings.data]

In [3]:
# cluster res on embeddings
cluster_num = 50

from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=cluster_num, random_state=0).fit(embeddings)
labels = kmeans.labels_
clusters = {}
for i, label in enumerate(labels):
    if label not in clusters:
        clusters[label] = []
    clusters[label].append(res[i])
for label, items in clusters.items():
    print(f"Cluster {label}:")
    for item in items[:10]:
        print(f"  - {item}")

KeyboardInterrupt: 

In [ ]:
import asyncio
from openai import AsyncOpenAI

model = "gpt-5-mini"


async def summarize_cluster(client: AsyncOpenAI, label, items):
    cluster_text = "\n".join(items[: min(50, len(items))])  # use at most 50 items
    prompt = f"Summarize all following processes into one short phrase no more than 10 words (e.g. document processing):\n{cluster_text}\nSummary:"

    response = await client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        prompt_cache_key="agent_skill_use",
    )

    summary = response.choices[0].message.content.strip()
    print(f"Cluster {label} summary: {summary}")

    return {
        "cluster": label,
        "items": items,
        "summary": summary,
    }


client = AsyncOpenAI()

tasks = [summarize_cluster(client, label, items) for label, items in clusters.items()]

res = await asyncio.gather(*tasks)

In [ ]:
summaries = []
for item in res:
    summaries.append(item["summary"])

In [ ]:
# ask a model to further summarize the categories

from openai import OpenAI

model = "gpt-5-mini"
client = OpenAI()

all_summaries = ",".join(summaries)
prompt = f"Summarize the following categories into broader categories:\n{all_summaries}"
response = client.chat.completions.create(
    model=model,
    messages=[{"role": "user", "content": prompt}],
)
final_summary = response.choices[0].message.content.strip()
print("Final summary of categories:")
print(final_summary)

In [ ]:
# save res to json
import json
for item in res:
    item["cluster"] = int(item["cluster"])
with open("cluster_summaries.json", "w") as f:
    json.dump(res, f, indent=2)
    